In [ ]:
import numpy as np
import pandas as pd
from datetime import datetime
from ortools.constraint_solver import routing_enums_pb2
from ortools.constraint_solver import pywrapcp

def create_data_model():
    data = {}
    
    # Nomes reais dos locais (Nó 0 é o Depósito)
    data['location_names'] = [
        "Carga (Depósito Central)",
        "CARLOS RODRIGUES-SUPERMERCADOS | SUPERMERCADO OÁSIS",
        "A.M.FERREIRA A. E SILVA,LDA-LI | AV. MORAIS SOARES, 137-139",
        "GLOVO - MFC ESTEVÃO | R. TOMAS ALCAIDE 46",
        "MERCEARIA DA PRATA | EMP. PRATA RIVERSIDE VILLAGE",
        "CASH POUPANCA MARVILA | AV. INFANTE D. HENRIQUE - ED.",
        "DERLIA SUPERMERCADOS, LDA. | ALAMEDA DOS OCEANOS LT. 4.48.0",
        "CNS CAMPUS NEUROLOGICOCO | BAIRRO DE SANTO ANTONIO Nº47",
        "LUIS PEDRO VALENTE PEREIRA LOP | R. DR. JOSE BASTOS , 12",
        "GLOVO - MFC ESTEVÃO | R. TOMAS ALCAIDE 46 (PEDIDO 2)",
        "A.M.FERREIRA A. E SILVA,LDA-LI | AV. MORAIS SOARES, 137-139 (PEDIDO 2)",
        "MERCEARIA DA PRATA | EMP. PRATA RIVERSIDE VILLAGE (PEDIDO 2)"
    ]
    
    # Coordenadas geográficas estimadas (X, Y em km) relativas ao depósito (0,0)
    coordinates = [
        (0.0, 0.0),    # 0. Depósito
        (2.5, 3.1),    # 1. Supermercado Oásis
        (-1.8, -4.2),  # 2. Av. Morais Soares 1
        (4.1, 1.2),    # 3. R. Tomás Alcaide 1
        (4.8, -0.8),   # 4. Prata Riverside 1
        (4.3, -2.1),   # 5. Av. Infante D. Henrique
        (5.9, 3.8),    # 6. Al. dos Oceanos
        (14.5, 18.2),  # 7. CNS Bairro Santo António 
        (13.8, 17.5),  # 8. R. Dr. José Bastos 
        (4.1, 1.2),    # 9. R. Tomás Alcaide 2
        (-1.8, -4.2),  # 10. Av. Morais Soares 2
        (4.8, -0.8)    # 11. Prata Riverside 2
    ]
    
    # Construção da matriz de distâncias reais com base nas coordenadas
    num_locations = len(coordinates)
    distance_matrix = np.zeros((num_locations, num_locations), dtype=int)
    for i in range(num_locations):
        for j in range(num_locations):
            dist = np.hypot(coordinates[i][0] - coordinates[j][0], coordinates[i][1] - coordinates[j][1])
            distance_matrix[i][j] = int(dist * 1.3) # Fator de circuito urbano
            
    data['distance_matrix'] = distance_matrix.tolist()
    data['time_matrix'] = (distance_matrix * 1.5).astype(int).tolist() # 1.5 min por km
    
    # Configurações operacionais da frota e procura
    data['demands'] = [0, 2, 1, 2, 2, 1, 2, 3, 2, 1, 1, 1]  
    data['vehicle_capacities'] = [6, 6, 6, 6, 6]            # 5 Camiões disponíveis
    data['num_vehicles'] = 5
    data['depot'] = 0
    
    # Janelas horárias operacionais em minutos (A partir das 08:00)
    data['time_windows'] = [
        (0, 480), (30, 180), (15, 120), (40, 240), (60, 300), (20, 150),
        (90, 360), (60, 240), (45, 240), (40, 240), (15, 120), (60, 300)
    ]
    data['service_times'] = [0, 15, 15, 12, 20, 15, 15, 25, 20, 12, 15, 20]
    
    return data

def main():
    data = create_data_model()
    
    manager = pywrapcp.RoutingIndexManager(len(data['distance_matrix']), data['num_vehicles'], data['depot'])
    routing = pywrapcp.RoutingModel(manager)

    # Registo dos Callbacks do motor matemático
    def distance_callback(from_index, to_index):
        return data['distance_matrix'][manager.IndexToNode(from_index)][manager.IndexToNode(to_index)]
    transit_callback_index = routing.RegisterTransitCallback(distance_callback)
    routing.SetArcCostEvaluatorOfAllVehicles(transit_callback_index)

    def time_callback(from_index, to_index):
        from_node = manager.IndexToNode(from_index)
        to_node = manager.IndexToNode(to_index)
        return data['time_matrix'][from_node][to_node] + data['service_times'][from_node]
    time_callback_index = routing.RegisterTransitCallback(time_callback)

    def demand_callback(from_index):
        return data['demands'][manager.IndexToNode(from_index)]
    demand_callback_index = routing.RegisterUnaryTransitCallback(demand_callback)

    # --- REGRAS E RESTRIÇÕES ---
    
    # 1. Restrição de Capacidade de Carga
    routing.AddDimensionWithVehicleCapacity(demand_callback_index, 0, data['vehicle_capacities'], True, 'Capacity')
    
    # 2. Restrição de Janelas Horárias (60 min de tolerância para espera ativa)
    routing.AddDimension(time_callback_index, 60, 480, True, 'Time')
    time_dimension = routing.GetDimensionOrDie('Time')
    for location_node, time_window in enumerate(data['time_windows']):
        if location_node == 0: continue
        index = manager.NodeToIndex(location_node)
        time_dimension.CumulVar(index).SetRange(time_window[0], time_window[1])

    # 3. Limite de 80km para viabilizar a periferia
    routing.AddDimension(transit_callback_index, 0, 80, True, 'Distance')
    
    # 4. Restrição de paragens (Máximo 4 entregas por veículo)
    routing.AddDimension(routing.RegisterTransitCallback(lambda f, t: 1), 0, 5, True, 'Stops')

    # Configuração dos algoritmos de busca combinatória
    search_parameters = pywrapcp.DefaultRoutingSearchParameters()
    search_parameters.first_solution_strategy = routing_enums_pb2.FirstSolutionStrategy.PATH_CHEAPEST_ARC
    search_parameters.local_search_metaheuristic = routing_enums_pb2.LocalSearchMetaheuristic.GUIDED_LOCAL_SEARCH
    search_parameters.time_limit.seconds = 5

    solution = routing.SolveWithParameters(search_parameters)

    if solution:
        build_and_display_table(data, manager, routing, solution)
    else:
        print("Inviável: Verifique se existem regras contraditórias nos parâmetros.")

def build_and_display_table(data, manager, routing, solution):
    time_dimension = routing.GetDimensionOrDie('Time')
    distance_dimension = routing.GetDimensionOrDie('Distance')
    
    rows = []
    data_rota = datetime.now().strftime("%d/%m/%Y")
    matriculas_frota = ["44-XF-12", "88-TG-94", "11-QA-54", "33-VB-88", "99-PL-01"]
    
    for vehicle_id in range(data['num_vehicles']):
        index = routing.Start(vehicle_id)
        end_index = routing.End(vehicle_id)
        
        if solution.Value(distance_dimension.CumulVar(end_index)) == 0:
            continue
            
        rota_id = vehicle_id + 1
        transportador = "TFS"
        matricula = matriculas_frota[vehicle_id]
        
        while not routing.IsEnd(index):
            node_index = manager.IndexToNode(index)
            
            time_var = time_dimension.CumulVar(index)
            dist_var = distance_dimension.CumulVar(index)
            
            chegada_min = solution.Min(time_var)
            
            # A linha que causou o erro foi corrigida aqui:
            if node_index != 0:
                partida_min = chegada_min + data['service_times'][node_index]
            else:
                partida_min = chegada_min + 30
                
            km_acumulados = solution.Value(dist_var)
            
            hora_prev_chegada = f"{8 + chegada_min // 60:02d}:{chegada_min % 60:02d}"
            hora_prev_saida = f"{8 + partida_min // 60:02d}:{partida_min % 60:02d}"
            
            nome_ponto = data['location_names'][node_index]
            
            rows.append({
                "Data": data_rota,
                "Rota": rota_id,
                "Transportador": transportador,
                "Matricula": matricula,
                "Locais": nome_ponto,
                "Horário previsto Chegada": hora_prev_chegada if node_index != 0 else "08:00",
                "Horário Real Chegada": "", 
                "Horário previsto Saida": hora_prev_saida,
                "Horário real saida": "",    
                "km": km_acumulados
            })
            
            index = solution.Value(routing.NextVar(index))
            
        node_index = manager.IndexToNode(index)
        time_var = time_dimension.CumulVar(index)
        dist_var = distance_dimension.CumulVar(index)
        
        chegada_min = solution.Min(time_var)
        km_totais = solution.Value(dist_var)
        hora_final_chegada = f"{8 + chegada_min // 60:02d}:{chegada_min % 60:02d}"
        
        rows.append({
            "Data": data_rota,
            "Rota": rota_id,
            "Transportador": transportador,
            "Matricula": matricula,
            "Locais": "Descarga/Fim (Depósito Central)",
            "Horário previsto Chegada": hora_final_chegada,
            "Horário Real Chegada": "",
            "Horário previsto Saida": "Turno Terminado",
            "Horário real saida": "",
            "km": km_totais
        })

    df = pd.DataFrame(rows)
    pd.set_option('display.max_rows', None)
    pd.set_option('display.max_columns', None)
    pd.set_option('display.width', 1000)
    
    display(df)

if __name__ == '__main__':
    main()

,Data,Rota,Transportador,Matricula,Locais,Horário previsto Chegada,Horário Real Chegada,Horário previsto Saida,Horário real saida,km
0,30/06/2026,2,TFS,88-TG-94,Carga (Depósito Central),08:00,,08:30,,0
1,30/06/2026,2,TFS,88-TG-94,CARLOS RODRIGUES-SUPERMERCADOS | SUPERMERCADO ...,08:30,,08:45,,5
2,30/06/2026,2,TFS,88-TG-94,"DERLIA SUPERMERCADOS, LDA. | ALAMEDA DOS OCEAN...",09:30,,09:45,,9
3,30/06/2026,2,TFS,88-TG-94,GLOVO - MFC ESTEVÃO | R. TOMAS ALCAIDE 46,09:51,,10:03,,13
4,30/06/2026,2,TFS,88-TG-94,Descarga/Fim (Depósito Central),10:10,,Turno Terminado,,18
5,30/06/2026,3,TFS,11-QA-54,Carga (Depósito Central),08:00,,08:30,,0
6,30/06/2026,3,TFS,11-QA-54,GLOVO - MFC ESTEVÃO | R. TOMAS ALCAIDE 46 (PED...,08:40,,08:52,,5
7,30/06/2026,3,TFS,11-QA-54,CNS CAMPUS NEUROLOGICOCO | BAIRRO DE SANTO ANT...,09:29,,09:54,,30
8,30/06/2026,3,TFS,11-QA-54,LUIS PEDRO VALENTE PEREIRA LOP | R. DR. JOSE B...,09:55,,10:15,,31
9,30/06/2026,3,TFS,11-QA-54,Descarga/Fim (Depósito Central),10:57,,Turno Terminado,,59
